# Prueba rápida (1 minuto)

Corre la celda. Debe terminar diciendo **FUNCIONA**. Si dice NO FUNCIONA, copia todo lo que imprime.

In [ ]:
import sys
import numpy as np
import pygimli as pg
import pygimli.meshtools as mt
from pygimli.physics import ert

print("Python", sys.version.split()[0], "| PyGIMLi", pg.__version__,
      "| NumPy", np.__version__)

world = mt.createWorld(start=[-50, 0], end=[50, -50], layers=[-1, -7],
                       worldMarker=True)
geom = world + mt.createCircle(pos=[0, -3], radius=1.5, nSegments=48, marker=4,
                               boundaryMarker=10, area=0.05)
scheme = ert.createData(elecs=np.linspace(-15, 15, 21), schemeName="dd")
for p in scheme.sensors():
    geom.createNode(p)
    geom.createNode(p - [0, 0.1])
mesh = mt.createMesh(geom, quality=34)
data = ert.simulate(mesh, scheme=scheme,
                    res=[[1, 100], [2, 50], [3, 200], [4, 500]],
                    noiseLevel=1, noiseAbs=1e-6, seed=1337, verbose=False)
data.remove(data["rhoa"] < 0)

mgr = ert.ERTManager(data)
modelo = mgr.invert(data, lam=20, verbose=True)
chi2 = mgr.inv.chi2()
var = float(np.std(np.log(np.array(modelo))))

print("\nchi2 final = %.2f   (variacion del modelo = %.3f)" % (chi2, var))
if chi2 < 3 and var > 0.05:
    print("=" * 50 + "\n   FUNCIONA: la inversion converge\n" + "=" * 50)
else:
    print("=" * 50 + "\n   NO FUNCIONA: la inversion no avanzo\n" + "=" * 50)